# 01 — Input Validation and Physics Spec

Validate the State A and State B artifacts, inspect the load case and selector hints, and surface the baseline FEA replication config.

In [ ]:
from pathlib import Path
import json
import logging
import os
import shutil
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code_base").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / "code_base" / "fea_cad_one_sample"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

import src.interfaces as api

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

sample_id = "00689964"
OUTPUT_ROOT = MODULE_ROOT / "outputs"
STATE_A_DIR = OUTPUT_ROOT / f"sample_{sample_id}" / "01_dataset_original"
STATE_B_DIR = OUTPUT_ROOT / f"sample_{sample_id}" / "02_fea_constrained_revision"
LOAD_CASE_PATH = STATE_B_DIR / "load_case.json"
SELECTOR_HINTS_PATH = STATE_B_DIR / "selector_hints.json"
STATE_B_STEP = STATE_B_DIR / "fea_revision.step"

print("[SETUP] complete")
print(f"  → sample_id : {sample_id}")
print(f"  → state A   : {STATE_A_DIR}")
print(f"  → state B   : {STATE_B_DIR}")

In [ ]:
print("[STAGE] required artifacts")
required_paths = [
    STATE_A_DIR / "original_prompt.txt",
    STATE_A_DIR / "database_original_code.py",
    STATE_A_DIR / "original.step",
    STATE_A_DIR / "original.stl",
    LOAD_CASE_PATH,
    SELECTOR_HINTS_PATH,
    STATE_B_STEP,
    STATE_B_DIR / "fea_revision.stl",
]
for path in required_paths:
    print(f"  → {path}")
    assert path.exists(), f"missing required artifact: {path}"
assert len(required_paths) == 8

In [ ]:
print("[STAGE] load-case and physics spec")
load_case = api.LoadCase(**json.loads(LOAD_CASE_PATH.read_text(encoding="utf-8")))
selector_hints = api.SelectorHints(**json.loads(SELECTOR_HINTS_PATH.read_text(encoding="utf-8")))
config = api.build_baseline_config(
    run_dir=STATE_B_DIR,
    source_step_path=STATE_B_STEP,
    mesh_size_mm=5.0,
    load_magnitude_n=200.0,
)
print(f"  ✓ load_case.sample_id : {load_case.sample_id}")
print(f"  ✓ load_case.material  : {load_case.material.get('name')}")
print(f"  ✓ load magnitude      : {load_case.loads[0].get('magnitude_n')} N")
print(f"  ✓ fixed selector      : {selector_hints.fixed_region_selector}")
print(f"  ✓ load selector       : {selector_hints.load_region_selector}")
print(f"  ✓ mesh size           : {config.mesh_size_mm} mm")
print(f"  ✓ output dir          : {config.run_dir}")
print(f"  ✓ load direction      : {config.load.direction_vector}")
print(f"  ✓ safety factor       : {config.verification_criteria.required_safety_factor}")
assert load_case.sample_id == sample_id
assert selector_hints.sample_id == sample_id
assert config.load.magnitude_n == 200.0
assert config.verification_criteria.required_safety_factor >= 2.0

In [ ]:
print("[STAGE] failure case")
try:
    missing_path = STATE_B_DIR / "definitely_missing.json"
    missing_path.read_text(encoding="utf-8")
except Exception as exc:
    print(f"  ✓ error type : {type(exc).__name__}")
    print(f"  ✓ message    : {exc}")

## Summary

- The notebook shows the canonical State A/B artifact roots.
- The load case, selector hints, and baseline physics config are visible without deep imports.
- The failure case makes the missing-artifact path explicit.